# 03 — Evaluate: Base vs Fine-tuned

**Purpose:** Compare the fine-tuned model against the base model on the **held-out eval split** from `01`, and log a comparison table + metrics to MLflow.

This notebook evaluates the LoRA adapter produced by `02a`/`02b`. To evaluate the Mosaic AI model from `02c`, set `FINE_TUNED_UC_MODEL` instead and load it from Unity Catalog.

> **Cluster:** single-node GPU (same as `02a`).

In [ ]:
%pip install -q -U transformers peft datasets mlflow evaluate rouge_score
dbutils.library.restartPython()

In [ ]:
# ─────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────
BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"

CATALOG = "main"
SCHEMA  = "otel_finetuning"
VOLUME  = f"/Volumes/{CATALOG}/{SCHEMA}/finetune"
EVAL_PATH = f"{VOLUME}/eval.jsonl"

# Which fine-tuned artifact to evaluate:
ADAPTER_PATH        = f"{VOLUME}/lora_adapter"   # from 02a (or lora_adapter_ray from 02b)
FINE_TUNED_UC_MODEL = None                       # set to "main.otel_finetuning.otel_sft_model" for 02c

MAX_NEW_TOKENS = 256
MLFLOW_EXPERIMENT = "/Shared/otel-finetuning"

## Load eval split

In [ ]:
from datasets import load_dataset

eval_ds = load_dataset("json", data_files=EVAL_PATH, split="train")

# Split each example into the prompt (system+user) and the reference answer.
def to_prompt_ref(ex):
    msgs = ex["messages"]
    prompt_msgs = [m for m in msgs if m["role"] in ("system", "user")]
    reference   = next(m["content"] for m in msgs if m["role"] == "assistant")
    return {"prompt_msgs": prompt_msgs, "reference": reference}

eval_pairs = [to_prompt_ref(e) for e in eval_ds]
print(f"{len(eval_pairs)} eval examples")

## Load base and fine-tuned models

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto")

# Fine-tuned = base + LoRA adapter (02a/02b). For 02c, load the UC model instead.
ft_model = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto"),
    ADAPTER_PATH)

## Generate completions

In [ ]:
def generate(model, prompt_msgs):
    inputs = tokenizer.apply_chat_template(
        prompt_msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(inputs, max_new_tokens=MAX_NEW_TOKENS,
                             do_sample=False, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

records = []
for pair in eval_pairs:
    records.append({
        "prompt": pair["prompt_msgs"][-1]["content"],
        "reference": pair["reference"],
        "base_output": generate(base_model, pair["prompt_msgs"]),
        "ft_output": generate(ft_model, pair["prompt_msgs"]),
    })
print("Generated", len(records), "comparisons")

## Score with ROUGE-L (reference overlap)

In [ ]:
import evaluate

rouge = evaluate.load("rouge")
base_score = rouge.compute(predictions=[r["base_output"] for r in records],
                           references=[r["reference"] for r in records])
ft_score   = rouge.compute(predictions=[r["ft_output"] for r in records],
                           references=[r["reference"] for r in records])

print(f"Base  ROUGE-L: {base_score['rougeL']:.4f}")
print(f"FT    ROUGE-L: {ft_score['rougeL']:.4f}")

> **On metrics:** ROUGE-L is a lightweight, dependency-free proxy for "did the output move toward the reference." For a production evaluation, prefer an **LLM-as-judge** via `mlflow.evaluate(..., model_type="databricks-agent")` or task-specific checks (e.g. does the generated SQL parse / execute).

## Log comparison to MLflow

In [ ]:
import mlflow
import pandas as pd

comparison_df = pd.DataFrame(records)

mlflow.set_experiment(MLFLOW_EXPERIMENT)
with mlflow.start_run(run_name="eval-base-vs-ft"):
    mlflow.log_metric("base_rougeL", base_score["rougeL"])
    mlflow.log_metric("ft_rougeL", ft_score["rougeL"])
    mlflow.log_metric("rougeL_delta", ft_score["rougeL"] - base_score["rougeL"])
    mlflow.log_table(comparison_df, artifact_file="comparison.json")

display(comparison_df)